<a href="https://colab.research.google.com/github/bhimappa-123/Bhima/blob/main/FeatureExtraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

one-hot-encoding

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
texts = [
    "I love NLP",
    "I love machine learning",
    "I love AI",
    "I love Deep Learning"
]

In [ ]:
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(texts)

In [ ]:
print(vectorizer.get_feature_names_out())
print(X.toarray())

['ai' 'deep' 'learning' 'love' 'machine' 'nlp']
[[0 0 0 1 0 1]
 [0 0 1 1 1 0]
 [1 0 0 1 0 0]
 [0 1 1 1 0 0]]


TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

texts = [
    "I love NLP",
    "I love machine learning",
    "I love AI",
    "I love Deep Learning"
]

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(texts)

print(vectorizer.get_feature_names_out())
print(X.toarray())

['ai' 'deep' 'learning' 'love' 'machine' 'nlp']
[[0.         0.         0.         0.46263733 0.         0.88654763]
 [0.         0.         0.5728925  0.37919167 0.72664149 0.        ]
 [0.88654763 0.         0.         0.46263733 0.         0.        ]
 [0.         0.72664149 0.5728925  0.37919167 0.         0.        ]]


Simple_RNN

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
corpus = [
    "I love machine learning",
    "word2vec is great algorithm",
    "Implementing word2vec is fun"
]

In [ ]:
sentences = [s.lower().split() for s in corpus]

In [ ]:
vocab = sorted(set(word for sent in sentences for word in sent))

In [ ]:
vocab

['algorithm',
 'fun',
 'great',
 'i',
 'implementing',
 'is',
 'learning',
 'love',
 'machine',
 'word2vec']

In [ ]:
word2idx = {w:i for i,w in enumerate(vocab)}
# idx2word = {i:w for w,i in word2idx.items()}
idx2word = {i:w for i,w in enumerate(vocab)}

In [ ]:
idx2word

{0: 'algorithm',
 1: 'fun',
 2: 'great',
 3: 'i',
 4: 'implementing',
 5: 'is',
 6: 'learning',
 7: 'love',
 8: 'machine',
 9: 'word2vec'}

In [ ]:
vocab_size = len(vocab)

In [ ]:
word2idx

{'algorithm': 0,
 'fun': 1,
 'great': 2,
 'i': 3,
 'implementing': 4,
 'is': 5,
 'learning': 6,
 'love': 7,
 'machine': 8,
 'word2vec': 9}

In [ ]:
X = []
Y = []
for sent in sentences:
    for i in range(len(sent)-1):
        X.append(word2idx[sent[i]])
        Y.append(word2idx[sent[i+1]])

In [ ]:
X = torch.tensor(X)
Y = torch.tensor(Y)

In [ ]:
print(X)
print(Y)

tensor([3, 7, 8, 9, 5, 2, 4, 9, 5])
tensor([7, 8, 6, 5, 2, 0, 9, 5, 1])


In [ ]:
XX = X.reshape(3,3)

In [ ]:
YY = Y.reshape(3,3)

In [ ]:
vocab_size

10

In [ ]:
class NextWordRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=16):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):x
        x = self.embedding(x) # [3, 3, 100]
        out, hx = self.rnn(x) # [3, 3, 16], [1, 3, 16]
        out = out[:, -1, :] # [3, 16]
        out = self.fc(out) # [3, 10]
        return out

In [ ]:
model = NextWordRNN(vocab_size)

In [ ]:
outputs = model(XX)

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [ ]:
for epoch in range(300):

    optimizer.zero_grad()

    inputs = X.unsqueeze(1)
    outputs = model(inputs)

    loss = loss_fn(outputs, Y)

    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print("Epoch:", epoch, "Loss:", loss.item())

Epoch: 0 Loss: 2.215301036834717
Epoch: 50 Loss: 0.17972414195537567
Epoch: 100 Loss: 0.16475027799606323
Epoch: 150 Loss: 0.1605737954378128
Epoch: 200 Loss: 0.15853577852249146
Epoch: 250 Loss: 0.15736337006092072


In [ ]:
def predict_next(word):
    model.eval()
    idx = torch.tensor([[word2idx[word.lower()]]])

    with torch.no_grad():
        out = model(idx)
        pred = torch.argmax(out).item()

    return idx2word[pred]

In [ ]:
print("machine=>", predict_next("machine"))
print("is=>", predict_next("is"))
print("word2vec=>", predict_next("word2vec"))


machine=> learning
is=> fun
word2vec=> is
